<a href="https://colab.research.google.com/github/5atish/TrainingCourse/blob/main/Hugging%20Face%20Agents%20Course%20/Unit%203.%20Use%20Case%20for%20Agentic%20RAG/Creating_Your_Gala_Agent.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [3]:
!pip install ddgs
!pip install rank_bm25
!pip install smolagents
!pip install langchain_groq
!pip install langchain_community
!pip install langchain_huggingface
!pip install llama-index-llms-groq
!pip install llama_index-llms

!pip install llama-index-tools-mcp
!pip install llama-index-llms-huggingface-api
!pip install llama-index-tools-duckduckgo
!pip install llama-index-tools-core
!pip install llama-index-embeddings-huggingface

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 161.5/161.5 kB 7.8 MB/s eta 0:00:00


In [29]:
!pip install langchain_groq

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 137.5/137.5 kB 7.0 MB/s eta 0:00:00


In [14]:
# Import necessary libraries
import random
from smolagents import CodeAgent, InferenceClientModel, Tool

# Import our custom tools from their modules
# from tools import DuckDuckGoSearchTool, WeatherInfoTool, HubStatsTool
# from retriever import load_guest_dataset

In [15]:
%%writefile tools.py
from smolagents import Tool
from huggingface_hub import list_models
import requests
import os

class SerperSearchTool(Tool):
    name = "web_search"
    description = "Searches the web using Google (via Serper) and returns the top result snippets."
    inputs = {
        "query": {"type": "string", "description": "The search query."}
    }
    output_type = "string"

    def forward(self, query: str):
        headers = {"X-API-KEY": os.environ["SERPER_API_KEY"], "Content-Type": "application/json"}
        response = requests.post("https://google.serper.dev/search", headers=headers, json={"q": query})
        results = response.json()
        if "organic" in results and results["organic"]:
            top = results["organic"][0]
            return f"{top.get('title', '')}: {top.get('snippet', 'No snippet available.')}"
        return "No results found."

DuckDuckGoSearchTool = SerperSearchTool  # alias — DuckDuckGo was unreliable, using Serper instead

class WeatherInfoTool(Tool):
    name = "weather_info"
    description = "Fetches weather information for a given location."
    inputs = {
        "location": {"type": "string", "description": "The location to get weather for."}
    }
    output_type = "string"

    def forward(self, location: str):
        return f"Weather lookup for {location} not yet implemented."

class HubStatsTool(Tool):
    name = "hub_stats"
    description = "Fetches the most downloaded model from a specific author on the Hugging Face Hub."
    inputs = {
        "author": {"type": "string", "description": "The username of the model author/organization."}
    }
    output_type = "string"

    def forward(self, author: str):
        try:
            models = list(list_models(author=author, sort="downloads", limit=1))
            if models:
                model = models[0]
                return f"The most downloaded model by {author} is {model.id} with {model.downloads:,} downloads."
            return f"No models found for author {author}."
        except Exception as e:
            return f"Error fetching models for {author}: {str(e)}"

Overwriting tools.py


In [16]:
%%writefile retriever.py
from smolagents import Tool
from langchain_community.retrievers import BM25Retriever
from langchain_core.documents import Document
from datasets import load_dataset

class GuestInfoRetrieverTool(Tool):
    name = "guest_info_retriever"
    description = "Retrieves detailed information about gala guests based on their name or relation."
    inputs = {
        "query": {"type": "string", "description": "The name or relation of the guest you want information about."}
    }
    output_type = "string"

    def __init__(self, docs, **kwargs):
        super().__init__(**kwargs)
        self.retriever = BM25Retriever.from_documents(docs)

    def forward(self, query: str):
        results = self.retriever.invoke(query)
        if results:
            return "\n\n".join([doc.page_content for doc in results[:3]])
        return "No matching guest information found."

def load_guest_dataset():
    guest_dataset = load_dataset("agents-course/unit3-invitees", split="train")
    docs = [
        Document(
            page_content="\n".join([
                f"Name: {g['name']}",
                f"Relation: {g['relation']}",
                f"Description: {g['description']}",
                f"Email: {g['email']}",
            ]),
            metadata={"name": g["name"]}
        )
        for g in guest_dataset
    ]
    return GuestInfoRetrieverTool(docs)

Overwriting retriever.py


In [17]:
import sys
sys.path.insert(0, '/content')
for mod in ['tools', 'retriever']:
    if mod in sys.modules:
        del sys.modules[mod]

from google.colab import userdata
import os
os.environ["GROQ_API_KEY"] = userdata.get('GROQ_API_KEY')
os.environ["SERPER_API_KEY"] = userdata.get('SERPER_API_KEY')

from smolagents import CodeAgent, OpenAIServerModel
from tools import DuckDuckGoSearchTool, WeatherInfoTool, HubStatsTool
from retriever import load_guest_dataset

model = OpenAIServerModel(
    model_id="openai/gpt-oss-120b",
    api_base="https://api.groq.com/openai/v1",
    api_key=os.environ["GROQ_API_KEY"]
)

search_tool = DuckDuckGoSearchTool()
weather_info_tool = WeatherInfoTool()
hub_stats_tool = HubStatsTool()
guest_info_tool = load_guest_dataset()

alfred = CodeAgent(
    tools=[guest_info_tool, weather_info_tool, hub_stats_tool, search_tool],
    model=model,
    add_base_tools=True,
    planning_interval=3
)

response = alfred.run("Tell me about our guest named 'Lady Ada Lovelace'.")
print(response)

╭──────────────────────────────────────────────────── New run ────────────────────────────────────────────────────╮
│                                                                                                                 │
│ Tell me about our guest named 'Lady Ada Lovelace'.                                                              │
│                                                                                                                 │
╰─ OpenAIModel - openai/gpt-oss-120b ─────────────────────────────────────────────────────────────────────────────╯

────────────────────────────────────────────────── Initial plan ───────────────────────────────────────────────────
Here are the facts I know and the plan of action that I will follow to solve the task:
```
**1. Facts survey**

### 1.1. Facts given in the task
- The guest’s name is **“Lady Ada Lovelace.”**

### 1.2. Facts to look up
| Fact needed | Where to find it |
|-------------|------------------|
| Basic biographical data (full name, birth/death dates, nationality, notable achievements) | 
`guest_info_retriever("Lady Ada Lovelace")` |
| Specific role or relationship to our gala/event (e.g., invited speaker, honorary guest) | 
`guest_info_retriever("Lady Ada Lovelace")` (if the system stores event‑specific metadata) |
| Additional details not stored in the guest database (e.g., historical context, legacy, famous quotes) | 
`web_search("Ada Lovelace biography")` then `visit_webpage` on the top result(s) |
| Any recent news or recent mentions of “Lady Ada Lovelace” in the context of our organization (if applicable) | 
`web_search("Lady Ada Lovelace" + [organization name])` |

### 1.3. Facts to derive
- A concise, readable summary that combines the retrieved biographical information with any event‑specific details 
(e.g., why she is a guest, what she will contribute).  
- Identify any notable achievements or quotes that are relevant to highlighting her significance for the audience. 

---

**2. Plan**

1. Query the **guest information retriever** with the name “Lady Ada Lovelace” to obtain any stored profile data, 
including biographical basics and event‑specific role.  
2. Evaluate the retrieved profile:  
   - If it contains a complete biography and event role, proceed to step 5.  
   - If the profile is missing key biographical details or event context, continue to step 3.  
3. Perform a **web search** for “Ada Lovelace biography” to locate authoritative sources (e.g., Wikipedia, 
biography.com).  
4. **Visit** the top search result(s) to extract the needed factual data (full name, birth/death dates, 
nationality, major contributions, notable quotes).  
5. Synthesize the collected information into a coherent paragraph that:  
   - Introduces Lady Ada Lovelace with her full name and dates.  
   - Highlights her most important achievements (e.g., first computer programmer, work on Charles Babbage’s 
Analytical Engine).  
   - States her specific connection to our event (e.g., invited speaker on computational history).  
   - Optionally includes a memorable quote.  
6. Return the synthesized summary via **final_answer**.  

```

━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ Step 1 ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

Error in generating model output:
Error code: 400 - {'error': {'message': 'Tool choice is none, but model called a tool', 'type': 
'invalid_request_error', 'code': 'tool_use_failed', 'failed_generation': '{"name": "code", "arguments": 
<code>\nprofile = guest_info_retriever(query="Lady Ada Lovelace")\nprint(profile)\n}'}}

[Step 1: Duration 0.83 seconds]

AgentGenerationError: Error in generating model output:
Error code: 400 - {'error': {'message': 'Tool choice is none, but model called a tool', 'type': 'invalid_request_error', 'code': 'tool_use_failed', 'failed_generation': '{"name": "code", "arguments": <code>\nprofile = guest_info_retriever(query="Lady Ada Lovelace")\nprint(profile)\n}'}}

In [18]:
# Initialize the Hugging Face model
model = InferenceClientModel()

# Initialize the web search tool
search_tool = DuckDuckGoSearchTool()

# Initialize the weather tool
weather_info_tool = WeatherInfoTool()

# Initialize the Hub stats tool
hub_stats_tool = HubStatsTool()

# Load the guest dataset and initialize the guest info tool
guest_info_tool = load_guest_dataset()

# Create Alfred with all the tools
alfred = CodeAgent(
    tools=[guest_info_tool, weather_info_tool, hub_stats_tool, search_tool],
    model=model,
    add_base_tools=True,  # Add any additional base tools
    planning_interval=3   # Enable planning every 3 steps
)

In [19]:
from llama_index.core.agent.workflow import AgentWorkflow
from llama_index.llms.groq import Groq
from llama_tools import search_tool, weather_info_tool, hub_stats_tool
from llama_retriever import guest_info_tool

In [20]:
%%writefile llama_tools.py
from llama_index.core.tools import FunctionTool
from huggingface_hub import list_models
import requests
import os

def serper_search(query: str) -> str:
    """Search the web using Google (via Serper) and return the top result snippet."""
    headers = {"X-API-KEY": os.environ["SERPER_API_KEY"], "Content-Type": "application/json"}
    response = requests.post("https://google.serper.dev/search", headers=headers, json={"q": query})
    results = response.json()
    if "organic" in results and results["organic"]:
        top = results["organic"][0]
        return f"{top.get('title', '')}: {top.get('snippet', 'No snippet available.')}"
    return "No results found."

def get_weather_info(location: str) -> str:
    """Fetches weather information for a given location."""
    return f"Weather lookup for {location} not yet implemented."

def get_hub_stats(author: str) -> str:
    """Fetches the most downloaded model from a specific author on the Hugging Face Hub."""
    try:
        models = list(list_models(author=author, sort="downloads", limit=1))
        if models:
            model = models[0]
            return f"The most downloaded model by {author} is {model.id} with {model.downloads:,} downloads."
        return f"No models found for author {author}."
    except Exception as e:
        return f"Error fetching models for {author}: {str(e)}"

search_tool = FunctionTool.from_defaults(serper_search)
weather_info_tool = FunctionTool.from_defaults(get_weather_info)
hub_stats_tool = FunctionTool.from_defaults(get_hub_stats)

Overwriting llama_tools.py


In [21]:
%%writefile llama_retriever.py
from llama_index.core.tools import FunctionTool
from langchain_community.retrievers import BM25Retriever
from langchain_core.documents import Document
from datasets import load_dataset

def load_guest_dataset():
    guest_dataset = load_dataset("agents-course/unit3-invitees", split="train")
    docs = [
        Document(
            page_content="\n".join([
                f"Name: {g['name']}",
                f"Relation: {g['relation']}",
                f"Description: {g['description']}",
                f"Email: {g['email']}",
            ]),
            metadata={"name": g["name"]}
        )
        for g in guest_dataset
    ]
    retriever = BM25Retriever.from_documents(docs)

    def get_guest_info(query: str) -> str:
        """Retrieves detailed information about gala guests based on their name or relation."""
        results = retriever.invoke(query)
        if results:
            return "\n\n".join([doc.page_content for doc in results[:3]])
        return "No matching guest information found."

    return FunctionTool.from_defaults(get_guest_info)

guest_info_tool = load_guest_dataset()

Overwriting llama_retriever.py


In [22]:
import sys
sys.path.insert(0, '/content')
for mod in ['llama_tools', 'llama_retriever']:
    if mod in sys.modules:
        del sys.modules[mod]

from google.colab import userdata
import os
os.environ["GROQ_API_KEY"] = userdata.get('GROQ_API_KEY')
os.environ["SERPER_API_KEY"] = userdata.get('SERPER_API_KEY')

from llama_index.core.agent.workflow import AgentWorkflow
from llama_index.llms.groq import Groq
from llama_tools import search_tool, weather_info_tool, hub_stats_tool
from llama_retriever import guest_info_tool

llm = Groq(
    model="openai/gpt-oss-120b",
    api_key=os.environ["GROQ_API_KEY"]
)

alfred = AgentWorkflow.from_tools_or_functions(
    [guest_info_tool, weather_info_tool, hub_stats_tool, search_tool],
    llm=llm
)

response = await alfred.run("Tell me about our guest named 'Lady Ada Lovelace'.")
print(response)

Here’s the information we have on **Lady Ada Lovelace**:

- **Name:** Ada Lovelace  
- **Relation to you:** Best friend  
- **Description:** Lady Ada Lovelace is an esteemed mathematician and a close confidante. She is celebrated for her pioneering work on Charles Babbage’s Analytical Engine and is often regarded as the world’s first computer programmer. Her insights into the potential of machines to go beyond mere calculation make her a fascinating conversational partner, especially if you’re interested in the history of computing or the intersection of mathematics and art.  
- **Email:** ada.lovelace@example.com  

If you’d like to arrange a meeting, send her a message, or need any additional details (e.g., preferred topics of conversation, upcoming availability, etc.), just let me know!


In [23]:
# Initialize the Hugging Face model
llm = HuggingFaceInferenceAPI(model_name="Qwen/Qwen2.5-Coder-32B-Instruct")

# Create Alfred with all the tools
alfred = AgentWorkflow.from_tools_or_functions(
    [guest_info_tool, search_tool, weather_info_tool, hub_stats_tool],
    llm=llm,
)

In [26]:
from typing import TypedDict, Annotated
from langgraph.graph.message import add_messages
from langchain_core.messages import AnyMessage, HumanMessage, AIMessage
from langgraph.prebuilt import ToolNode
from langgraph.graph import START, StateGraph
from langgraph.prebuilt import tools_condition
from langchain_huggingface import HuggingFaceEndpoint, ChatHuggingFace
from langchain_community.tools import DuckDuckGoSearchRun

# from tools import DuckDuckGoSearchRun, weather_info_tool, hub_stats_tool
# from retriever import guest_info_tool

from llama_index.core.agent.workflow import AgentWorkflow
from llama_index.llms.groq import Groq
from llama_tools import search_tool, weather_info_tool, hub_stats_tool
from llama_retriever import guest_info_tool

In [27]:
# # Initialize the web search tool
# search_tool = DuckDuckGoSearchRun()

# # Generate the chat interface, including the tools
# llm = HuggingFaceEndpoint(
#     repo_id="Qwen/Qwen2.5-Coder-32B-Instruct",
#     huggingfacehub_api_token=HUGGINGFACEHUB_API_TOKEN,
# )

# chat = ChatHuggingFace(llm=llm, verbose=True)
# tools = [guest_info_tool, search_tool, weather_info_tool, hub_stats_tool]
# chat_with_tools = chat.bind_tools(tools)

# # Generate the AgentState and Agent graph
# class AgentState(TypedDict):
#     messages: Annotated[list[AnyMessage], add_messages]

# def assistant(state: AgentState):
#     return {
#         "messages": [chat_with_tools.invoke(state["messages"])],
#     }

# ## The graph
# builder = StateGraph(AgentState)

# # Define nodes: these do the work
# builder.add_node("assistant", assistant)
# builder.add_node("tools", ToolNode(tools))

# # Define edges: these determine how the control flow moves
# builder.add_edge(START, "assistant")
# builder.add_conditional_edges(
#     "assistant",
#     # If the latest message requires a tool, route to tools
#     # Otherwise, provide a direct response
#     tools_condition,
# )
# builder.add_edge("tools", "assistant")
# alfred = builder.compile()

NameError: name 'HUGGINGFACEHUB_API_TOKEN' is not defined

In [31]:
from langchain_groq import ChatGroq
from langchain_core.tools import tool
from langchain_community.retrievers import BM25Retriever
from langchain_core.documents import Document
from datasets import load_dataset
from langgraph.graph import StateGraph, START
from langgraph.graph.message import add_messages
from langgraph.prebuilt import ToolNode, tools_condition
from langchain_core.messages import AnyMessage
from typing import TypedDict, Annotated
from google.colab import userdata
from huggingface_hub import list_models
import requests
import os

os.environ["GROQ_API_KEY"] = userdata.get('GROQ_API_KEY')
os.environ["SERPER_API_KEY"] = userdata.get('SERPER_API_KEY')

@tool
def search_tool(query: str) -> str:
    """Search the web using Google (via Serper) and return the top result snippet."""
    headers = {"X-API-KEY": os.environ["SERPER_API_KEY"], "Content-Type": "application/json"}
    response = requests.post("https://google.serper.dev/search", headers=headers, json={"q": query})
    results = response.json()
    if "organic" in results and results["organic"]:
        top = results["organic"][0]
        return f"{top.get('title', '')}: {top.get('snippet', 'No snippet available.')}"
    return "No results found."

@tool
def weather_info_tool(location: str) -> str:
    """Fetches weather information for a given location."""
    return f"Weather lookup for {location} not yet implemented."

@tool
def hub_stats_tool(author: str) -> str:
    """Fetches the most downloaded model from a specific author on the Hugging Face Hub."""
    try:
        models = list(list_models(author=author, sort="downloads", limit=1))
        if models:
            model = models[0]
            return f"The most downloaded model by {author} is {model.id} with {model.downloads:,} downloads."
        return f"No models found for author {author}."
    except Exception as e:
        return f"Error fetching models for {author}: {str(e)}"

guest_dataset = load_dataset("agents-course/unit3-invitees", split="train")
docs = [
    Document(
        page_content="\n".join([
            f"Name: {g['name']}",
            f"Relation: {g['relation']}",
            f"Description: {g['description']}",
            f"Email: {g['email']}",
        ]),
        metadata={"name": g["name"]}
    )
    for g in guest_dataset
]
retriever = BM25Retriever.from_documents(docs)

@tool
def guest_info_tool(query: str) -> str:
    """Retrieves detailed information about gala guests based on their name or relation."""
    results = retriever.invoke(query)
    if results:
        return "\n\n".join([doc.page_content for doc in results[:3]])
    return "No matching guest information found."

chat_with_tools_base = ChatGroq(
    model="openai/gpt-oss-120b",
    groq_api_key=os.environ["GROQ_API_KEY"],
)

tools = [guest_info_tool, search_tool, weather_info_tool, hub_stats_tool]
chat_with_tools = chat_with_tools_base.bind_tools(tools)

class AgentState(TypedDict):
    messages: Annotated[list[AnyMessage], add_messages]

def assistant(state: AgentState):
    return {"messages": [chat_with_tools.invoke(state["messages"])]}

builder = StateGraph(AgentState)
builder.add_node("assistant", assistant)
builder.add_node("tools", ToolNode(tools))
builder.add_edge(START, "assistant")
builder.add_conditional_edges("assistant", tools_condition)
builder.add_edge("tools", "assistant")
alfred = builder.compile()

In [32]:
from langchain_core.messages import HumanMessage

messages = [HumanMessage(content="Tell me about our guest named 'Lady Ada Lovelace'.")]
response = alfred.invoke({"messages": messages})

print("🎩 Alfred's Response:")
print(response['messages'][-1].content)

🎩 Alfred's Response:
**Guest Profile – Lady Ada Lovelace**

| Detail | Information |
|--------|--------------|
| **Name** | **Ada Lovelace** (Lady Ada Lovelace) |
| **Relation to you** | Best friend |
| **Description** | Ada is an esteemed mathematician and is celebrated as the world’s first computer programmer for her pioneering notes on Charles Babbine’s Analytical Engine. Her keen intellect, curiosity about emerging technologies, and warm personality make her a delightful presence at any gathering. |
| **Contact** | **Email:** ada.lovelace@example.com |

If you’d like to arrange a special welcome, discuss a particular topic with her, or need any additional details (e.g., dietary preferences, travel plans, etc.), just let me know!


In [34]:
# from langchain_core.messages import HumanMessage

query = "Tell me about Lady Ada Lovelace. What's her background?"
messages = [HumanMessage(content=query)]
response = alfred.invoke({"messages": messages})

print("🎩 Alfred's Response:")
print(response['messages'][-1].content)

🎩 Alfred's Response:
**Ada Lovelace (Augusta Ada King, Countess of Lovelace)**  
*Born:* 10 December 1815, London, England  
*Died:* 27 November 1852, Marylebone, London, England  

---

## 1. Family and Early Life  

| Aspect | Details |
|--------|---------|
| **Parents** | • **Father:** *George Gordon, 4th Earl of Warrington* (later *George Lord Lovelace*). A distinguished British aristocrat and politician.<br>• **Mother:** *Ann Isabella Milbanke*, a mathematically‑inclined aristocrat known as the “Princess of Parallelograms.” She insisted Ada receive a strong education in mathematics to counteract what she feared was her father’s “madness.” |
| **Birth name** | *Augusta Ada Byron* (she was the only legitimate child of the poet **Lord Byron**). |
| **Siblings** | No full siblings; she had a half‑brother, **Alfred Lord Byron**, from her father’s earlier marriage. |
| **Childhood environment** | Raised in an elite London household, Ada was exposed to both the literary world (through he

In [35]:
# from langchain_core.messages import HumanMessage

query = "What's the weather like in Paris tonight? Will it be suitable for our fireworks display?"
messages = [HumanMessage(content=query)]
response = alfred.invoke({"messages": messages})

print("🎩 Alfred's Response:")
print(response['messages'][-1].content)

🎩 Alfred's Response:
I’m not able to pull a real‑time forecast for Paris at the moment, so I can’t give you a definitive answer about tonight’s conditions. 

**What to look for when deciding whether a fireworks display will be safe and effective**

| Factor | Why it matters | Ideal range for fireworks |
|--------|----------------|---------------------------|
| **Cloud cover** | Low clouds can obscure the display and make it hard for the audience to see. | Mostly clear or only thin high‑altitude clouds. |
| **Precipitation** | Rain or drizzle can damp the shells, causing misfires or duds. | No rain (dry conditions). |
| **Wind speed** | Strong winds can carry sparks and debris farther than intended, posing a fire hazard and blowing smoke into the audience. | ≤ 10 km/h (≈ 6 mph) at ground level; calmer is better. |
| **Humidity** | Very high humidity can affect the burn rate of some pyrotechnic compositions, though it’s less critical than rain. | Moderate (40 %–70 %). |
| **Temperature**

In [36]:
# from langchain_core.messages import HumanMessage

query = "One of our guests is from Qwen. What can you tell me about their most popular model?"
messages = [HumanMessage(content=query)]
response = alfred.invoke({"messages": messages})

print("🎩 Alfred's Response:")
print(response['messages'][-1].content)

🎩 Alfred's Response:
**Qwen / Qwen3‑0.6B**  
*Most‑downloaded model from the Qwen organization on Hugging Face (≈22.5 M downloads)*  

---

### Quick Overview
| Property | Details |
|----------|---------|
| **Model name** | `Qwen/Qwen3-0.6B` |
| **Architecture** | Decoder‑only transformer (similar to LLaMA‑style) |
| **Parameters** | ~0.6 billion (600 M) |
| **Training data** | Massive multilingual corpus (≈2 trillion tokens) covering 100+ languages, with a strong emphasis on English, Chinese, and code. |
| **Pre‑training objective** | Causal language modeling with instruction‑following finetuning (RLHF‑style) |
| **Licensing** | Apache 2.0 (permissive, commercial‑friendly) |
| **Quantization options** | Available in FP16, Q4_K_M, Q5_K_S, and GGUF formats for edge deployment |
| **Typical inference speed** | ~150 tokens/s on an RTX 3090 (FP16); ~300 tokens/s on a modern CPU with GGUF 4‑bit quantization |
| **Peak performance** | Comparable to a 2‑B‑parameter model on English benchmarks

In [37]:
# from langchain_core.messages import HumanMessage

query = "One of our guests is from Google. What can you tell me about their most popular model?"
messages = [HumanMessage(content=query)]
response = alfred.invoke({"messages": messages})

print("🎩 Alfred's Response:")
print(response['messages'][-1].content)

🎩 Alfred's Response:
**Google’s most‑downloaded model on Hugging Face: `google/electra‑base‑discriminator`**  

| Item | Details |
|------|---------|
| **Model family** | **ELECTRA** (Efficiently Learning an Encoder that Classifies Token Replacements) |
| **Specific checkpoint** | `google/electra-base-discriminator` |
| **Size** | 12 Transformer layers, 768 hidden units, ~110 M parameters (the “base” configuration) |
| **Training objective** | **Discriminator** in the ELECTRA pre‑training scheme – the model learns to predict whether each token in a corrupted input has been replaced by a generator model. This is a *replaced‑token detection* task, which is more sample‑efficient than the traditional masked‑language‑model (MLM) objective used by BERT. |
| **Pre‑training data** | • **BooksCorpus** (≈800 M words) <br>• **English Wikipedia** (≈2.5 B words) <br>• **OpenWebText** (filtered web crawl) <br>• **CC‑News** (Common Crawl news articles) <br>All data are lower‑cased and tokenized with 

In [38]:
query = "I need to speak with Dr. Nikola Tesla about recent advancements in wireless energy. Can you help me prepare for this conversation?"
messages = [HumanMessage(content=query)]
response = alfred.invoke({"messages": messages})

print("🎩 Alfred's Response:")
print(response['messages'][-1].content)

🎩 Alfred's Response:
Below is a concise briefing you can use to structure your conversation with “Dr. Nikola Tesla” (or anyone familiar with his work) about the latest breakthroughs in wireless energy. It’s organized into three sections:

1. **Quick Tesla refresher** – his original vision and the key experiments that still inspire today’s research.  
2. **State‑of‑the‑art wireless‑power technologies (2024‑2026)** – the most promising approaches, recent milestones, and commercial pilots.  
3. **Suggested talking points & questions** – how to bridge Tesla’s legacy with today’s reality and spark an engaging dialogue.

---

## 1. Tesla in a nutshell (for context)

| Year | Milestone | Relevance to modern wireless power |
|------|-----------|------------------------------------|
| **1887‑1888** | Developed the **alternating‑current (AC) system** and the **Tesla coil** (high‑frequency, high‑voltage resonant transformer). | The Tesla coil introduced **resonant inductive coupling**, the founda

In [39]:
from smolagents import CodeAgent, OpenAIServerModel
from tools import DuckDuckGoSearchTool, WeatherInfoTool, HubStatsTool
from retriever import load_guest_dataset
from google.colab import userdata
import os

os.environ["GROQ_API_KEY"] = userdata.get('GROQ_API_KEY')
os.environ["SERPER_API_KEY"] = userdata.get('SERPER_API_KEY')

model = OpenAIServerModel(
    model_id="openai/gpt-oss-120b",
    api_base="https://api.groq.com/openai/v1",
    api_key=os.environ["GROQ_API_KEY"]
)

search_tool = DuckDuckGoSearchTool()  # aliased to SerperSearchTool in tools.py
weather_info_tool = WeatherInfoTool()
hub_stats_tool = HubStatsTool()
guest_info_tool = load_guest_dataset()

# Create Alfred with conversation memory
alfred_with_memory = CodeAgent(
    tools=[guest_info_tool, weather_info_tool, hub_stats_tool, search_tool],
    model=model,
    add_base_tools=True,
    planning_interval=3
)

# First interaction
response1 = alfred_with_memory.run("Tell me about Lady Ada Lovelace.")
print("🎩 Alfred's First Response:")
print(response1)

# Second interaction (referencing the first)
response2 = alfred_with_memory.run("What projects is she currently working on?", reset=False)
print("🎩 Alfred's Second Response:")
print(response2)

╭──────────────────────────────────────────────────── New run ────────────────────────────────────────────────────╮
│                                                                                                                 │
│ Tell me about Lady Ada Lovelace.                                                                                │
│                                                                                                                 │
╰─ OpenAIModel - openai/gpt-oss-120b ─────────────────────────────────────────────────────────────────────────────╯

────────────────────────────────────────────────── Initial plan ───────────────────────────────────────────────────
Here are the facts I know and the plan of action that I will follow to solve the task:
```
**1. Facts survey**

### 1.1. Facts given in the task
- The subject to be described is *Lady Ada Lovelace* (also known as Ada Byron, Countess of Lovelace).

### 1.2. Facts to look up
| Fact needed | Reason for need | Likely source(s) |
|------------|----------------|------------------|
| Full name (including maiden name) | To provide accurate identification | Wikipedia page “Ada Lovelace”, 
Biography.com |
| Birth date and place | Basic biographical detail | Wikipedia, Encyclopedia Britannica |
| Death date and place | Complete life span | Same as above |
| Parents (especially father) | Context of upbringing and influence | Wikipedia, biographies |
| Education and mentors (e.g., Charles Babbage, Mary Somerville) | Explain intellectual formation | Wikipedia, 
scholarly articles |
| Major works (e.g., “Notes” on the Analytical Engine, first algorithm) | Core contribution to computing | 
Wikipedia, original 1843 “Notes” (available via archive.org) |
| Date of publication of the “Notes” (1843) | Timeline of achievement | Wikipedia, original paper |
| Description of the Analytical Engine | To convey the machine she wrote for | Wikipedia, historical texts |
| Recognition (e.g., first computer programmer, honors, statues) | Modern significance | Wikipedia, news articles, 
museum sites |
| Personal life (marriage to William King, children) | Human interest & context | Wikipedia, biography sources |
| Later influence on computer science and popular culture | Relevance today | Books on computing history, articles,
museum exhibits |
| Any notable quotes | Illustrative material | Collected works, quotations databases |

**Where to find each:**  
- **Web search** for “Ada Lovelace biography”, “Ada Lovelace birth date”, “Ada Lovelace analytical engine notes”.  
- **Visit webpages** such as the Wikipedia article (`https://en.wikipedia.org/wiki/Ada_Lovelace`), the British 
Library’s Ada Lovelace collection, and reputable encyclopedia sites.  
- **Visit web page** of the original 1843 paper (e.g., `https://archive.org/details/NotesOnTheAnalyticalEngine`) 
for primary‑source verification.

### 1.3. Facts to derive
- A concise, chronological narrative of Ada Lovelace’s life using the retrieved dates and events.  
- An explanation of why her notes are considered the first computer program (derive from the description of the 
algorithm for Bernoulli numbers).  
- A summary of her lasting impact on the field of computing (synthesizing modern recognitions and citations).  
- Any notable contradictions or myths to clarify (e.g., “first programmer” debates).  

---

**2. Plan**

1. **Search for authoritative biographical sources** on Ada Lovelace (e.g., Wikipedia, Britannica, reputable 
biographies) using `web_search`.  
2. **Visit the top result(s)** to extract structured data: full name, birth/death dates & places, parents, 
education, mentors, marriage, children.  
3. **Search for and retrieve details** on her collaboration with Charles Babbage and the 1843 “Notes” (including 
the algorithm for Bernoulli numbers) via `web_search` and `visit_webpage`.  
4. **Collect information** on the Analytical Engine’s description to contextualize her work.  
5. **Search for modern recognitions** (statues, awards, “first computer programmer” label) and notable quotes using
`web_search` and `visit_webpage`.  
6. **Synthesize** the gathered facts into a coherent narrative: early life → education & influences → work on the 
Analytical Engine → publication of the Notes → later life & legacy.  
7. **Derive** explanations of her significance (why the algorithm is considered the first program) and summarize 
her impact on contemporary computing.  
8. **Prepare the final answer** summarizing all the derived information in a clear, well‑structured format.  


```

━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ Step 1 ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

Error in generating model output:
Error code: 400 - {'error': {'message': 'Tool choice is none, but model called a tool', 'type': 
'invalid_request_error', 'code': 'tool_use_failed', 'failed_generation': '{"name": "code", "arguments": 
<code>\nsearch_results = web_search(query="Ada Lovelace Wikipedia")\nprint(search_results)\n}'}}

[Step 1: Duration 1.02 seconds]

AgentGenerationError: Error in generating model output:
Error code: 400 - {'error': {'message': 'Tool choice is none, but model called a tool', 'type': 'invalid_request_error', 'code': 'tool_use_failed', 'failed_generation': '{"name": "code", "arguments": <code>\nsearch_results = web_search(query="Ada Lovelace Wikipedia")\nprint(search_results)\n}'}}

In [41]:
import sys
sys.path.insert(0, '/content')
for mod in ['llama_tools', 'llama_retriever']:
    if mod in sys.modules:
        del sys.modules[mod]

from google.colab import userdata
import os
os.environ["GROQ_API_KEY"] = userdata.get('GROQ_API_KEY')
os.environ["SERPER_API_KEY"] = userdata.get('SERPER_API_KEY')

from llama_index.core.agent.workflow import AgentWorkflow
from llama_index.core.workflow import Context
from llama_index.llms.groq import Groq
from llama_tools import search_tool, weather_info_tool, hub_stats_tool
from llama_retriever import guest_info_tool

llm = Groq(
    model="openai/gpt-oss-120b",
    api_key=os.environ["GROQ_API_KEY"]
)

alfred = AgentWorkflow.from_tools_or_functions(
    [guest_info_tool, search_tool, weather_info_tool, hub_stats_tool],
    llm=llm
)

# Remembering state
ctx = Context(alfred)

# First interaction
response1 = await alfred.run("Tell me about Lady Ada Lovelace.", ctx=ctx)
print("🎩 Alfred's First Response:")
print(response1)

# Second interaction (referencing the first)
response2 = await alfred.run("What projects is she currently working on?", ctx=ctx)
print("🎩 Alfred's Second Response:")
print(response2)

🎩 Alfred's First Response:
**Ada Lovelace (1815‑1852)**  
*Often called “the world’s first computer programmer,” Ada Lovelace is celebrated for her visionary work on Charles Babbage’s early mechanical computer, the Analytical Engine.*

---

## 1. Early Life & Background  

| Year | Event |
|------|-------|
| **10 December 1815** | Born Augusta Ada Byron in London, England, the only legitimate child of the poet **Lord George Gordon Byron** (the famous Romantic poet) and **Annabella Milbanke**, a mathematically‑inclined aristocrat. |
| **1816–1825** | After her parents separated, Ada lived with her mother, who encouraged her to study mathematics and logic to counteract what she feared were her father’s “poetic” tendencies. |
| **1828 (age 13)** | Met **Charles Babbage**, a mathematician and inventor, at a party. Babbage’s description of his “Difference Engine” fascinated her and sparked a lifelong interest in computing. |
| **1833–1835** | Studied under the mathematician **Mary Somervill

In [43]:
from langchain_groq import ChatGroq
from langchain_core.tools import tool
from langchain_community.retrievers import BM25Retriever
from langchain_core.documents import Document
from langchain_core.messages import HumanMessage, AnyMessage
from datasets import load_dataset
from langgraph.graph import StateGraph, START
from langgraph.graph.message import add_messages
from langgraph.prebuilt import ToolNode, tools_condition
from typing import TypedDict, Annotated
from google.colab import userdata
from huggingface_hub import list_models
import requests
import os

os.environ["GROQ_API_KEY"] = userdata.get('GROQ_API_KEY')
os.environ["SERPER_API_KEY"] = userdata.get('SERPER_API_KEY')

@tool
def search_tool(query: str) -> str:
    """Search the web using Google (via Serper) and return the top result snippet."""
    headers = {"X-API-KEY": os.environ["SERPER_API_KEY"], "Content-Type": "application/json"}
    response = requests.post("https://google.serper.dev/search", headers=headers, json={"q": query})
    results = response.json()
    if "organic" in results and results["organic"]:
        top = results["organic"][0]
        return f"{top.get('title', '')}: {top.get('snippet', 'No snippet available.')}"
    return "No results found."

@tool
def weather_info_tool(location: str) -> str:
    """Fetches weather information for a given location."""
    return f"Weather lookup for {location} not yet implemented."

@tool
def hub_stats_tool(author: str) -> str:
    """Fetches the most downloaded model from a specific author on the Hugging Face Hub."""
    try:
        models = list(list_models(author=author, sort="downloads", limit=1))
        if models:
            model = models[0]
            return f"The most downloaded model by {author} is {model.id} with {model.downloads:,} downloads."
        return f"No models found for author {author}."
    except Exception as e:
        return f"Error fetching models for {author}: {str(e)}"

guest_dataset = load_dataset("agents-course/unit3-invitees", split="train")
docs = [
    Document(
        page_content="\n".join([
            f"Name: {g['name']}",
            f"Relation: {g['relation']}",
            f"Description: {g['description']}",
            f"Email: {g['email']}",
        ]),
        metadata={"name": g["name"]}
    )
    for g in guest_dataset
]
retriever = BM25Retriever.from_documents(docs)

@tool
def guest_info_tool(query: str) -> str:
    """Retrieves detailed information about gala guests based on their name or relation."""
    results = retriever.invoke(query)
    if results:
        return "\n\n".join([doc.page_content for doc in results[:3]])
    return "No matching guest information found."

chat_with_tools_base = ChatGroq(
    model="openai/gpt-oss-120b",
    groq_api_key=os.environ["GROQ_API_KEY"],
)

tools = [guest_info_tool, search_tool, weather_info_tool, hub_stats_tool]
chat_with_tools = chat_with_tools_base.bind_tools(tools)

class AgentState(TypedDict):
    messages: Annotated[list[AnyMessage], add_messages]

def assistant(state: AgentState):
    return {"messages": [chat_with_tools.invoke(state["messages"])]}

builder = StateGraph(AgentState)
builder.add_node("assistant", assistant)
builder.add_node("tools", ToolNode(tools))
builder.add_edge(START, "assistant")
builder.add_conditional_edges("assistant", tools_condition)
builder.add_edge("tools", "assistant")
alfred = builder.compile()

# First interaction
response = alfred.invoke({"messages": [HumanMessage(content="Tell me about 'Lady Ada Lovelace'. What's her background and how is she related to me?")]})
print("🎩 Alfred's Response:")
print(response['messages'][-1].content)
print()

# Second interaction (referencing the first)
response = alfred.invoke({"messages": response["messages"] + [HumanMessage(content="What projects is she currently working on?")]})
print("🎩 Alfred's Response:")
print(response['messages'][-1].content)

🎩 Alfred's Response:
**Ada Lovelace – a quick snapshot**

| Item | Details |
|------|---------|
| **Full name** | **Augusta Ada King, Countess of Lovelace** (née Byron) |
| **Born** | 10 December 1815, London, England |
| **Died** | 27 November 1852 (aged 36), London, England |
| **Parents** | Poet **Lord George Gordon Byron** (the “Lord Byron”) and **Ann Isabella Milbanke** (a mathematician‑inclined “Princess of Parallelograms”). |
| **Education** | Tutored privately in mathematics and logic by Augustus De Morgan, Mary Somerville, and others; self‑taught in the emerging field of mechanical computation. |
| **Key achievement** | In 1843 she wrote the **first published algorithm** (for Charles Babbage’s Analytical Engine) and a visionary commentary that foresaw computers as more than number‑crunchers. This makes her widely regarded as the **world’s first computer programmer**. |
| **Legacy** | • The **Ada programming language** (1980s) is named after her.<br>• Numerous awards, scholarsh